# 20 — CART Global Surrogate Model — BPIC17

Trains an interpretable decision tree (CART) to approximate the LSTM's
next-activity predictions across all inputs. Reveals **model-level** decision
logic — which features and thresholds drive the model's behaviour globally.

Uses engineered aggregate features (prefix-level statistics) rather than
raw per-position values.

BPIC17 is large (253K entries) so we sample ~10K prefixes for the surrogate.

In [ ]:
import sys
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import torch
import numpy as np
import pandas as pd

# --- Load dataset ---
data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
dataset = torch.load(data_path, weights_only=False)

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f'Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}')

# --- Load model ---
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

model_path = (_current / 'src' / 'notebooks' / 'training_variational_dropout'
              / 'BPIC17' / 'BPIC_2017_full_grad_norm_new_4layer.pkl')
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(model_path), dropout=0.0)
model.eval()

# --- TensorDecoder + activity vocab ---
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(dataset)

ACTIVITY_FEATURE = 'concept:name'
activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

# --- Predictor ---
from src.interpretability.perturbation_methods.revised_plus.revised_plus import RevisedPlusModelPredictor

predictor = RevisedPlusModelPredictor(model, suffix_step=0, activity_feature=ACTIVITY_FEATURE)

print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'Activity vocabulary ({len(activity_names)}), EOS={eos_idx}')
print(f'Cat features: {decoder.cat_features}')
print(f'Num features: {decoder.num_features}')

In [ ]:
# Feature indices (BPIC17)
ACT_IDX = decoder.cat_features.index('concept:name')       # 0
RES_IDX = decoder.cat_features.index('org:resource')        # 2
LC_IDX = decoder.cat_features.index('lifecycle:transition')  # 4
LG_IDX = decoder.cat_features.index('case:LoanGoal')        # 5
AT_IDX = decoder.cat_features.index('case:ApplicationType')  # 6

elapsed_num_idx = decoder.num_features.index('case_elapsed_time')
event_num_idx = decoder.num_features.index('event_elapsed_time')
amount_num_idx = decoder.num_features.index('case:RequestedAmount')
credit_num_idx = decoder.num_features.index('CreditScore')
day_num_idx = decoder.num_features.index('day_in_week')

print(f'Act={ACT_IDX}, Res={RES_IDX}, LC={LC_IDX}, LG={LG_IDX}, AT={AT_IDX}')
print(f'elapsed={elapsed_num_idx}, event={event_num_idx}, amount={amount_num_idx}, '
      f'credit={credit_num_idx}, day={day_num_idx}')


def extract_features(cat_tuple, num_tuple, decoder):
    """Extract engineered aggregate features from a single BPIC17 prefix."""
    act_tensor = cat_tuple[ACT_IDX]
    mask = act_tensor != 0
    prefix_len = int(mask.sum().item())
    if prefix_len == 0:
        return None

    nonzero = mask.nonzero(as_tuple=True)[0]
    start = nonzero[0].item()
    end = start + prefix_len

    acts = act_tensor[start:end]

    # Trailing repetitions of last activity
    last_act = acts[-1].item()
    repeated = 1
    for k in range(len(acts) - 2, -1, -1):
        if acts[k].item() == last_act:
            repeated += 1
        else:
            break

    # Time features (inverse-transformed)
    total_elapsed = decoder.decode_numerical_value(
        'case_elapsed_time', num_tuple[elapsed_num_idx][end - 1].item())
    last_event_dur = decoder.decode_numerical_value(
        'event_elapsed_time', num_tuple[event_num_idx][end - 1].item())
    event_durs = [decoder.decode_numerical_value(
        'event_elapsed_time', num_tuple[event_num_idx][j].item())
        for j in range(start, end)]
    mean_event_dur = float(np.mean(event_durs))

    # Resource features
    res_tensor = cat_tuple[RES_IDX]
    resources = res_tensor[start:end].tolist()
    n_unique_res = len(set(resources))

    # BPIC17-specific numerical features
    requested_amount = decoder.decode_numerical_value(
        'case:RequestedAmount', num_tuple[amount_num_idx][end - 1].item())
    credit_score = decoder.decode_numerical_value(
        'CreditScore', num_tuple[credit_num_idx][end - 1].item())
    day_in_week = decoder.decode_numerical_value(
        'day_in_week', num_tuple[day_num_idx][end - 1].item())

    return {
        'prefix_length': prefix_len,
        'last_activity': decoder.decode_categorical_value(ACT_IDX, acts[-1].item()),
        'first_activity': decoder.decode_categorical_value(ACT_IDX, acts[0].item()),
        'n_unique_activities': len(set(acts.tolist())),
        'last_activity_repeated': repeated,
        'total_elapsed_time': total_elapsed,
        'last_event_duration': last_event_dur,
        'mean_event_duration': mean_event_dur,
        'last_resource': decoder.decode_categorical_value(RES_IDX, res_tensor[end - 1].item()),
        'n_unique_resources': n_unique_res,
        'loan_goal': decoder.decode_categorical_value(LG_IDX, cat_tuple[LG_IDX][end - 1].item()),
        'application_type': decoder.decode_categorical_value(AT_IDX, cat_tuple[AT_IDX][end - 1].item()),
        'requested_amount': requested_amount,
        'credit_score': credit_score,
        'last_lifecycle': decoder.decode_categorical_value(LC_IDX, cat_tuple[LC_IDX][end - 1].item()),
        'day_in_week': day_in_week,
    }


# Quick test
sample_feats = extract_features(dataset[0][0], dataset[0][1], decoder)
print('Sample features:')
for k, v in sample_feats.items():
    print(f'  {k}: {v}')

In [ ]:
SAMPLE_CASES = 1500  # sample this many unique cases, then slice into prefixes

# Collect unique cases (keep entry with longest trace)
print('Scanning dataset for unique cases...')
cases = {}
for i in range(len(dataset)):
    cat_t, num_t, case_id = dataset[i]
    trace_len = int((cat_t[0] != 0).sum().item())
    if case_id not in cases or trace_len > cases[case_id][1]:
        cases[case_id] = (i, trace_len)
    if (i + 1) % 50000 == 0:
        print(f'  Scanned {i + 1}/{len(dataset)}, found {len(cases)} unique cases...')

print(f'Found {len(cases)} unique cases')

# Sample cases
rng = np.random.RandomState(42)
all_case_ids = list(cases.keys())
if len(all_case_ids) > SAMPLE_CASES:
    sampled_case_ids = list(rng.choice(all_case_ids, size=SAMPLE_CASES, replace=False))
else:
    sampled_case_ids = all_case_ids
print(f'Using {len(sampled_case_ids)} cases')

# Slice each case into prefixes of varying lengths.
# Strip trailing EOS so prefixes end with real activities.
feature_rows = []
sliced_cat_list = []
sliced_num_list = []

for count, case_id in enumerate(sampled_case_ids):
    ds_idx, trace_len = cases[case_id]
    cat_full, num_full, _ = dataset[ds_idx]
    src_start = seq_len - trace_len

    # Find useful trace length (strip trailing EOS)
    acts = cat_full[ACT_IDX][src_start:src_start + trace_len]
    useful_len = trace_len
    for j in range(trace_len - 1, -1, -1):
        if acts[j].item() == eos_idx:
            useful_len = j
        else:
            break
    if useful_len == 0:
        continue

    # Generate prefixes of length 1 .. useful_len
    for k in range(1, useful_len + 1):
        pad_len = seq_len - k
        cat_prefix = []
        for c in cat_full:
            t = torch.zeros_like(c)
            t[pad_len:] = c[src_start:src_start + k]
            cat_prefix.append(t)

        num_prefix = []
        for n in num_full:
            t = torch.zeros_like(n)
            t[pad_len:] = n[src_start:src_start + k]
            num_prefix.append(t)

        feats = extract_features(tuple(cat_prefix), tuple(num_prefix), decoder)
        if feats is not None:
            feature_rows.append(feats)
            sliced_cat_list.append(cat_prefix)
            sliced_num_list.append(num_prefix)

    if (count + 1) % 500 == 0:
        print(f'  {count + 1}/{len(sampled_case_ids)} cases, {len(feature_rows)} prefixes...')

X = pd.DataFrame(feature_rows)
N_total = len(X)
print(f'Generated {N_total} prefixes from {len(sampled_case_ids)} cases')
print(f'\nPrefix length distribution:')
print(X['prefix_length'].value_counts().sort_index())
display(X.describe(include='all'))

# --- Batch-predict LSTM labels ---
print(f'\nBatch-predicting with LSTM...')
cat_batch = [
    torch.stack([sliced_cat_list[i][ci] for i in range(N_total)])
    for ci in range(n_cat)
]
num_stacked = torch.stack([
    torch.stack(sliced_num_list[i], dim=-1)
    for i in range(N_total)
])

preds, probs = predictor.predict_batch(cat_batch, num_stacked, batch_size=128)
y = np.array([activity_names[p] for p in preds])

print(f'\nLSTM prediction distribution:')
unique, counts = np.unique(y, return_counts=True)
for cls, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
    print(f'  {cls}: {cnt} ({cnt / len(y) * 100:.1f}%)')

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

cat_cols = [c for c in X.columns if X[c].dtype == 'object']
num_cols = [c for c in X.columns if c not in cat_cols]
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'Numerical columns ({len(num_cols)}): {num_cols}')

ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = X.copy()
X_encoded[cat_cols] = ord_encoder.fit_transform(X[cat_cols].astype(str))

cat_mappings = {}
for i, col in enumerate(cat_cols):
    cat_mappings[col] = list(ord_encoder.categories_[i])
    print(f'  {col}: {len(cat_mappings[col])} categories')

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y)
print(f'\nTrain: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# min_samples_leaf: increase to reduce scatter (fewer, larger leaves)
MIN_SAMPLES_LEAF = 20

depths = [3, 4, 5, 6, 8, None]
results = []

for depth in depths:
    clf = DecisionTreeClassifier(
        max_depth=depth, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=42)
    clf.fit(X_train, y_train)
    train_fid = clf.score(X_train, y_train)
    test_fid = clf.score(X_test, y_test)
    n_leaves = clf.get_n_leaves()
    results.append({
        'max_depth': depth if depth is not None else 'None',
        'min_samples_leaf': MIN_SAMPLES_LEAF,
        'train_fidelity': f'{train_fid:.4f}',
        'test_fidelity': f'{test_fid:.4f}',
        'n_leaves': n_leaves,
        'model': clf,
    })
    print(f'depth={str(depth):>4s}  train={train_fid:.4f}  test={test_fid:.4f}  leaves={n_leaves}')

df_results = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in results])
display(df_results)

best_idx = 0
best_test = float(results[0]['test_fidelity'])
for i in range(1, len(results)):
    ti = float(results[i]['test_fidelity'])
    if ti > best_test + 0.02:
        best_idx = i
        best_test = ti

best = results[best_idx]
clf_best = best['model']
print(f'\nSelected: depth={best["max_depth"]} (test fidelity={best["test_fidelity"]}, '
      f'leaves={best["n_leaves"]})')

if float(best['test_fidelity']) < 0.60:
    print('\nWARNING: Fidelity below 60% -- surrogate may not be meaningful!')
elif float(best['test_fidelity']) < 0.70:
    print('\nNote: Fidelity below 70% -- interpret surrogate rules with caution.')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

y_pred = clf_best.predict(X_test)
fidelity = (y_pred == y_test).mean()

print('=' * 60)
print(f'  OVERALL FIDELITY (surrogate vs LSTM): {fidelity:.4f}')
print('=' * 60)
print()
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(set(y_test) | set(y_pred))
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(labels, fontsize=6)
ax.set_xlabel('Surrogate prediction')
ax.set_ylabel('LSTM prediction')
ax.set_title(f'Confusion Matrix: CART Surrogate vs LSTM (fidelity={fidelity:.3f})')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=5)
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import plot_tree, export_text
import re

feature_names = list(X_encoded.columns)

fig, ax = plt.subplots(figsize=(28, 14))
plot_tree(clf_best, feature_names=feature_names,
          class_names=clf_best.classes_.tolist(),
          filled=True, rounded=True, ax=ax, fontsize=6)
ax.set_title(f'CART Surrogate (depth={best["max_depth"]}, fidelity={best["test_fidelity"]})')
plt.tight_layout()
plt.show()

tree_text = export_text(clf_best, feature_names=feature_names)


def decode_tree_text(text, cat_cols, cat_mappings):
    """Replace ordinal split thresholds with actual category names."""
    decoded = []
    for line in text.split('\n'):
        out = line
        for col in cat_cols:
            pattern = rf'({re.escape(col)})\s*(<=|>)\s*([\d.]+)'
            m = re.search(pattern, line)
            if m:
                feat, op, thresh = m.group(1), m.group(2), float(m.group(3))
                cats = cat_mappings[feat]
                if op == '<=':
                    incl = [c for i, c in enumerate(cats) if i <= thresh]
                else:
                    incl = [c for i, c in enumerate(cats) if i > thresh]
                if len(incl) <= 8:
                    out += f'  [{feat} in {{{", ".join(incl)}}}]'
                else:
                    out += f'  [{feat} in {{{", ".join(incl[:6])}, ... ({len(incl)} total)}}]'
                break
        decoded.append(out)
    return '\n'.join(decoded)


print(decode_tree_text(tree_text, cat_cols, cat_mappings))

In [ ]:
importances = clf_best.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
}).sort_values('importance', ascending=False)

print('Feature importances (CART surrogate):')
display(importance_df)

fig, ax = plt.subplots(figsize=(10, 6))
imp_nz = importance_df[importance_df['importance'] > 0].copy()
ax.barh(imp_nz['feature'], imp_nz['importance'])
ax.set_xlabel('Gini Importance')
ax.set_title('CART Surrogate Feature Importance (BPIC17)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import _tree


def explain_decision_path(clf, sample, feature_names, cat_cols, cat_mappings):
    """Show the decision path for a single sample through the tree."""
    sample_df = sample.to_frame().T
    node_indicator = clf.decision_path(sample_df)
    node_ids = node_indicator.indices
    tree = clf.tree_

    steps = []
    for node_id in node_ids:
        if tree.feature[node_id] != _tree.TREE_UNDEFINED:
            feat_idx = tree.feature[node_id]
            feat_name = feature_names[feat_idx]
            threshold = tree.threshold[node_id]
            value = sample.iloc[feat_idx]
            direction = '<=' if value <= threshold else '>'

            step = f'{feat_name} = {value:.2f} {direction} {threshold:.2f}'

            if feat_name in cat_cols:
                cats = cat_mappings[feat_name]
                if direction == '<=':
                    incl = [c for i, c in enumerate(cats) if i <= threshold]
                else:
                    incl = [c for i, c in enumerate(cats) if i > threshold]
                preview = ', '.join(incl[:5])
                if len(incl) > 5:
                    preview += f', ... ({len(incl)} total)'
                step += f'  (categories: {preview})'
            steps.append(step)
    return steps


n_examples = 5
rng = np.random.RandomState(42)
example_positions = rng.choice(len(X_test), size=min(n_examples, len(X_test)), replace=False)

for pos in example_positions:
    sample = X_test.iloc[pos]
    lstm_label = y_test[pos]
    surrogate_pred = clf_best.predict(sample.to_frame().T)[0]
    match = 'MATCH' if lstm_label == surrogate_pred else 'MISMATCH'

    la_val = sample['last_activity']
    la_name = cat_mappings['last_activity'][int(la_val)] if 'last_activity' in cat_cols else la_val

    print(f'\n{"=" * 80}')
    print(f'LSTM: {lstm_label} | Surrogate: {surrogate_pred} [{match}]')
    print(f'prefix_len={sample["prefix_length"]:.0f}, last_activity={la_name}')

    path = explain_decision_path(clf_best, sample, feature_names, cat_cols, cat_mappings)
    for i, step in enumerate(path):
        print(f'  Step {i + 1}: {step}')
    print(f'  -> Prediction: {surrogate_pred}')